In [2]:
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 1. Load Data
df = pd.read_csv('Alzheimers Mice Data.csv')
df['AD_Status'] = df['AD_Status'].astype(str)
df['Treatment'] = df['Treatment'].astype(str)

def analyze_variable(dv):
    print(f"\n--- ANOVA for {dv} ---")
    formula = f'{dv} ~ C(AD_Status) * C(Treatment)'
    model = ols(formula, data=df).fit()

    # Assumption Check
    _, p_norm = stats.shapiro(model.resid)
    groups = [group[dv].values for name, group in df.groupby(['AD_Status', 'Treatment'])]
    _, p_lev = stats.levene(*groups)
    print(f"Normality p: {p_norm:.4f}, Levene p: {p_lev:.4f}")

    # ANOVA Table
    table = sm.stats.anova_lm(model, typ=2)
    table['eta_sq'] = table['sum_sq'] / sum(table['sum_sq'])
    print(table.round(4))

    # Post-hoc
    print(pairwise_tukeyhsd(df[dv], df['Treatment']).summary())
    print(pairwise_tukeyhsd(df[dv], df['AD_Status']).summary())

analyze_variable('Training')
analyze_variable('Memory')


--- ANOVA for Training ---
Normality p: 0.2214, Levene p: 0.8731
                           sum_sq    df       F  PR(>F)  eta_sq
C(AD_Status)                3.025   1.0  1.2161  0.2784  0.0252
C(Treatment)               28.275   3.0  3.7889  0.0197  0.2357
C(AD_Status):C(Treatment)   9.075   3.0  1.2161  0.3198  0.0756
Residual                   79.600  32.0     NaN     NaN  0.6635
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
     1      2      1.5  0.172 -0.4223  3.4223  False
     1      3      0.9 0.5931 -1.0223  2.8223  False
     1      4     -0.7 0.7612 -2.6223  1.2223  False
     2      3     -0.6 0.8347 -2.5223  1.3223  False
     2      4     -2.2 0.0196 -4.1223 -0.2777   True
     3      4     -1.6 0.1314 -3.5223  0.3223  False
----------------------------------------------------
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   l